# Model Cascading

## Learning Objectives
1. Understand two-tier and multi-tier cascading
2. Implement confidence-based routing
3. Analyze cost-accuracy trade-offs
4. Calibrate decision thresholds

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.isotonic import IsotonicRegression

np.random.seed(42)
print('Model cascading simulation')

## Level 1: Basic Confidence Routing

In [ ]:
np.random.seed(42)
num_test = 1000

# Small model outputs (less confident)
small_logits = np.random.randn(num_test, 10)
small_probs = np.exp(small_logits) / np.exp(small_logits).sum(axis=1, keepdims=True)
small_confidence = small_probs.max(axis=1)
small_preds = small_probs.argmax(axis=1)

# Large model outputs (more confident)
large_logits = small_logits + np.random.randn(num_test, 10) * 0.3
large_probs = np.exp(large_logits) / np.exp(large_logits).sum(axis=1, keepdims=True)
large_confidence = large_probs.max(axis=1)
large_preds = large_probs.argmax(axis=1)

# True labels
true_labels = np.random.randint(0, 10, num_test)

# Simple routing
confidence_threshold = 0.6
route_small = small_confidence >= confidence_threshold
cascade_preds = np.where(route_small, small_preds, large_preds)

# Metrics
small_only_acc = (small_preds == true_labels).mean()
large_only_acc = (large_preds == true_labels).mean()
cascade_acc = (cascade_preds == true_labels).mean()

pct_small = (route_small).mean() * 100
pct_large = (1 - route_small.mean()) * 100

print(f'Small model only: {small_only_acc:.3f}')
print(f'Large model only: {large_only_acc:.3f}')
print(f'Cascade: {cascade_acc:.3f}')
print(f'\nRouting: {pct_small:.1f}% small, {pct_large:.1f}% large')

# Cost analysis
cascade_cost = (pct_small / 100) * 1.0 + (pct_large / 100) * 20.0
print(f'Cost: {cascade_cost:.1f}x (baseline 20x)')
print(f'Savings: {(1 - cascade_cost/20.0)*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(small_confidence, bins=30, alpha=0.6, label='Small', color='coral', edgecolor='black')
axes[0].hist(large_confidence, bins=30, alpha=0.6, label='Large', color='steelblue', edgecolor='black')
axes[0].axvline(confidence_threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {confidence_threshold}')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Model Confidence Distributions')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

thresholds = np.linspace(0.1, 0.95, 20)
accuracies = []
costs = []

for thresh in thresholds:
    route = small_confidence >= thresh
    preds = np.where(route, small_preds, large_preds)
    acc = (preds == true_labels).mean()
    cost = route.mean() * 1.0 + (1 - route.mean()) * 20.0
    accuracies.append(acc)
    costs.append(cost)

ax2 = axes[1]
ax2_cost = ax2.twinx()

ax2.plot(thresholds, accuracies, marker='o', linewidth=2.5, color='steelblue', label='Accuracy')
ax2_cost.plot(thresholds, costs, marker='s', linewidth=2.5, color='coral', label='Cost')

ax2.set_xlabel('Confidence Threshold')
ax2.set_ylabel('Accuracy', color='steelblue')
ax2_cost.set_ylabel('Relative Cost', color='coral')
ax2.set_title('Threshold Calibration')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Level 2: Isotonic Regression Calibration

In [ ]:
n_calib = 500
n_test = 500

calib_confidences = small_confidence[:n_calib]
calib_correct = (small_preds[:n_calib] == true_labels[:n_calib]).astype(int)

test_confidences = small_confidence[n_calib:]
test_correct = (small_preds[n_calib:] == true_labels[n_calib:]).astype(int)

iso_reg = IsotonicRegression(out_of_bounds='clip')
iso_reg.fit(calib_confidences, calib_correct)

calib_probs = iso_reg.predict(calib_confidences)
test_probs = iso_reg.predict(test_confidences)

target_accuracy = 0.95
thresholds_calib = np.linspace(0.1, 0.95, 50)

best_threshold = None
best_acc = None
best_cost = None

for thresh in thresholds_calib:
    route_small = test_probs >= thresh
    preds_calib = np.where(route_small, small_preds[n_calib:], large_preds[n_calib:])
    acc_calib = (preds_calib == true_labels[n_calib:]).mean()
    cost_calib = route_small.mean() * 1.0 + (1 - route_small.mean()) * 20.0
    
    if best_threshold is None or (acc_calib >= target_accuracy - 0.01 and cost_calib < best_cost):
        best_threshold = thresh
        best_acc = acc_calib
        best_cost = cost_calib

print(f'Isotonic Regression Calibration:')
print(f'Optimal threshold: {best_threshold:.3f}')
print(f'Resulting accuracy: {best_acc:.3f}')
print(f'Resulting cost: {best_cost:.2f}x (baseline 20x)')
print(f'Cost savings: {(1 - best_cost/20.0)*100:.1f}%')
print(f'Small model usage: {(test_probs >= best_threshold).mean()*100:.1f}%')

## Real-World Examples

In [ ]:
# Example 1: RAG-style two-tier cascade
print('Two-Tier Cascading Results:')
print('-' * 60)
print(f'Accuracy: {best_acc:.3f}')
print(f'Small model usage: {(test_probs >= best_threshold).mean()*100:.1f}%')
print(f'Total cost: ${best_cost:.4f}')
print(f'Savings: {(1 - best_cost/20.0)*100:.1f}%')

# Example 2: Three-tier cascade
tiny_preds = np.random.randint(0, 10, num_test)
medium_preds = np.random.randint(0, 10, num_test)

thresh_1 = 0.5
thresh_2 = 0.7

route_tier1 = small_confidence >= thresh_1
route_tier2 = (small_confidence < thresh_1) & (small_confidence >= thresh_2)
route_tier3 = (small_confidence < thresh_2)

cascade_preds_3 = np.where(route_tier1, small_preds,
                           np.where(route_tier2, medium_preds, large_preds))

print(f'\nThree-Tier Cascade:')
print(f'  Tier 1: {route_tier1.mean()*100:.1f}%')
print(f'  Tier 2: {route_tier2.mean()*100:.1f}%')
print(f'  Tier 3: {route_tier3.mean()*100:.1f}%')

## Key Takeaways

In [ ]:
print('='*70)
print('MODEL CASCADING BEST PRACTICES')
print('='*70)
print('')
print('1. THRESHOLD CALIBRATION')
print('   - Use isotonic regression on held-out calibration set')
print('   - Target SLA-based thresholds, not random percentiles')
print('   - Re-calibrate on data distribution shifts')
print('')
print('2. TWO-TIER VS THREE-TIER')
print('   - 2-tier: Simple, 3-5x cost savings, 1-2% accuracy loss')
print('   - 3-tier: More complex, 10x+ savings with tuning')
print('   - Diminishing returns: 4+ tiers rarely justified')
print('')
print('3. EXPECTED COST REDUCTIONS')
print('   - Typical: 70-80% cost savings at ~95% baseline accuracy')
print('   - Formula: cost = p_small * C_small + (1-p_small) * C_large')
print('')
print('4. WHEN TO USE CASCADE')
print('   - High accuracy + cost constraints')
print('   - Variable request difficulty')
print('   - Significant model size/cost gap')
print('='*70)

## Exercises

In [ ]:
print('Exercise 1: Threshold Optimization')
print('-' * 70)
print('Find threshold T such that: 80% use small model, cascade acc>=94%')
print('Use isotonic regression + threshold sweep')
print('')
print('Exercise 2: Dynamic Threshold Adjustment')
print('-' * 70)
print('Adjust thresholds based on current load:')
print('  - Low load: lower threshold → use more cheap model')
print('  - High load: higher threshold → preserve quality')
print('')
print('Success! Generated notebook 48 (model-cascading)')